# Importacao das libs

In [42]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.neighbors import NearestNeighbors
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
from pyspark import StorageLevel

# Configuração de dispositivo PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Criação de monitoramento com MlFLow

In [43]:
import mlflow
from mlflow.models import infer_signature
# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment('tech-challenge-recomendacao-retailrocket')

<Experiment: artifact_location='file:///c:/Users/lara-/workspace/projeto-tech-challenge-sistema-recomendacao/notebooks/mlruns/1', creation_time=1781996088566, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781996088566, lifecycle_stage='active', name='tech-challenge-recomendacao-retailrocket', tags={}, trace_location=None, workspace='default'>

In [44]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("RetailRocket")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "48")

In [45]:
# ==============================================================================
# 1. LEITURA, PREPARAÇÃO DOS DADOS E GERAÇÃO DO DF_EDA (PySpark)
# ==============================================================================

df_events = spark.read.csv("../data/interim/events.csv", header=True, inferSchema=True)
df_prop_1 = spark.read.csv("../data/interim/item_properties_part1.csv", header=True, inferSchema=True)
df_prop_2 = spark.read.csv("../data/interim/item_properties_part2.csv", header=True, inferSchema=True)

df_prop = df_prop_1.unionByName(df_prop_2)

# Janela para capturar a propriedade mais recente do item
window_spec = Window.partitionBy("itemid", "property").orderBy(F.col("timestamp").desc())

df_prop_last = (
    df_prop
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Pivota para obter categoryid e available
df_items = (
    df_prop_last
    .filter(F.col("property").isin(["categoryid", "available"]))
    .groupBy("itemid")
    .pivot("property")
    .agg(F.first("value"))
)

# Cruzamento dos eventos com os metadados dos itens
df_eda = df_events.join(df_items, on="itemid", how="left")

# Atribuição da coluna de pesos baseada nos eventos
WEIGHTS_EXPR = (
    F.when(F.col("event") == "view", F.lit(1))
     .when(F.col("event") == "addtocart", F.lit(3))
     .when(F.col("event") == "transaction", F.lit(5))
     .otherwise(F.lit(0))
)
df_eda_com_pesos = df_eda.withColumn("weight", WEIGHTS_EXPR)

In [46]:
# ==============================================================================
# 2. SPLIT TEMPORAL E FILTRO DE USUÁRIOS ATIVOS (PySpark)
# ==============================================================================

# Divisão de 80% para treino e 20% para teste baseado no tempo
cutoff = df_eda_com_pesos.approxQuantile("timestamp", [0.8], 0.01)[0]

df_treino_raw = df_eda_com_pesos.filter(F.col("timestamp") <= cutoff)
df_teste_raw  = df_eda_com_pesos.filter(F.col("timestamp") >  cutoff)

# Identificação de usuários com 5 ou mais interações no treino
contagem_treino = (
    df_treino_raw
    .groupBy("visitorid")
    .agg(F.countDistinct("itemid").alias("n_itens"))
)

usuarios_ativos = (
    contagem_treino
    .filter(F.col("n_itens") >= 5)
    .select("visitorid")
    .cache()
)

# Filtragem de usuários ativos em ambas as bases (Treino e Teste)
df_treino_spark = df_treino_raw.join(usuarios_ativos, "visitorid")
df_teste_spark  = df_teste_raw.join(usuarios_ativos, "visitorid")

# Dicionário do Ground Truth para validação de métricas
ground_truth = (
    df_teste_spark
    .groupBy("visitorid")
    .agg(F.collect_set("itemid").alias("itens_relevantes"))
)

gt_dict = {
    row["visitorid"]: row["itens_relevantes"]
    for row in ground_truth.collect()
}

In [47]:
# ==============================================================================
# 3. MAPEAMENTO DE INDICES SEQUENCIAIS (Importante para Camadas de Embedding)
# ==============================================================================

# Para que o PyTorch não estoure os limites das matrizes de Embedding, precisamos
# remapear visitorid, itemid e categoryid para índices sequenciais começando em 0.

# Mapeando Usuários
unique_users = [row['visitorid'] for row in df_treino_spark.select('visitorid').distinct().collect()]
user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}

# Mapeando Itens
unique_items = [row['itemid'] for row in df_eda_com_pesos.select('itemid').distinct().collect()]
item_to_idx = {iid: idx for idx, iid in enumerate(unique_items)}
idx_to_item = {idx: iid for idx, iid in enumerate(unique_items)}

# Mapeando Categorias (Preenchimento de nulos com categoria genérica se houver)
df_eda_com_pesos = df_eda_com_pesos.fillna({"categoryid": "unknown"})
unique_cats = [row['categoryid'] for row in df_eda_com_pesos.select('categoryid').distinct().collect()]
cat_to_idx = {cid: idx for idx, cid in enumerate(unique_cats)}

num_users = len(unique_users)
num_items = len(unique_items)
num_cats = len(unique_cats)

# Criando o vetor estático de mapeamento: item_idx -> cat_idx
item_cat_df = df_eda_com_pesos.select("itemid", "categoryid").distinct().toPandas()
item_to_category_vector = np.zeros(num_items, dtype=np.int64)

for _, row in item_cat_df.iterrows():
    if row['itemid'] in item_to_idx:
        i_idx = item_to_idx[row['itemid']]
        c_idx = cat_to_idx[row['categoryid']]
        item_to_category_vector[i_idx] = c_idx

# Calculando a popularidade dos itens (necessária para a amostragem negativa ponderada)
pop_df = df_treino_spark.groupBy("itemid").count().toPandas()
item_popularity = {item_to_idx[r['itemid']]: r['count'] for _, r in pop_df.iterrows() if r['itemid'] in item_to_idx}

In [48]:
# ==============================================================================
# 4. CONVERSÃO E PREPARAÇÃO DO DATAFRAME EM PANDAS (Para o PyTorch)
# ==============================================================================

df_train_pandas = df_treino_spark.select("visitorid", "itemid", "weight").toPandas()

# Substituição dos IDs originais pelos índices sequenciais criados
df_train_pandas['user_idx'] = df_train_pandas['visitorid'].map(user_to_idx)
df_train_pandas['item_idx'] = df_train_pandas['itemid'].map(item_to_idx)

# Remove possíveis linhas que ficaram sem mapeamento por consistência
df_train_pandas = df_train_pandas.dropna(subset=['user_idx', 'item_idx']).astype({'user_idx': 'int64', 'item_idx': 'int64'})


In [49]:
# ==============================================================================
# 5. DATASET VETORIZADO PARA CPU (Remoção de loops lentos)
# ==============================================================================

class RetailRocketBPRDatasetCPU(Dataset):
    """Dataset otimizado para CPU.
    Sorteia itens negativos de forma inteiramente vetorizada no __init__
    evitando chamadas repetidas a funções aleatórias durante o treino.
    """
    def __init__(self, df_interactions, item_popularity, n_items, power=0.75):
        self.users = torch.tensor(df_interactions['user_idx'].values, dtype=torch.long)
        self.pos_items = torch.tensor(df_interactions['item_idx'].values, dtype=torch.long)
        self.weights = torch.tensor(df_interactions['weight'].values, dtype=torch.float)
        
        # Pré-calcula as probabilidades de popularidade
        pop_weights = np.array([item_popularity.get(i, 0) for i in range(n_items)])
        pop_weights = np.power(pop_weights, power)
        pop_sum = pop_weights.sum()
        pop_probs = pop_weights / pop_sum if pop_sum > 0 else np.ones(n_items) / n_items
        
        # Sorteia TODOS os negativos de uma só vez via NumPy (Extremamente rápido na CPU)
        print("Pré-gerando amostras negativas para otimizar a CPU...")
        self.neg_items = np.random.choice(n_items, size=len(self.users), p=pop_probs)
        
        # Correção rápida caso o negativo coincida com o positivo
        mask_igual = (self.neg_items == df_interactions['item_idx'].values)
        if mask_igual.any():
            substitutos = np.random.choice(n_items, size=mask_igual.sum(), p=pop_probs)
            self.neg_items[mask_igual] = substitutos
            
        self.neg_items = torch.tensor(self.neg_items, dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.pos_items[idx], self.neg_items[idx], self.weights[idx]

In [50]:
# ==============================================================================
# 6. MODELO NEUMF ENXUTO (Menos neurônios = Menos custo computacional na CPU)
# ==============================================================================

class NeuMF_RetailRocket_CPU(nn.Module):
    def __init__(self, n_users: int, n_items: int, n_categorias: int, item_to_cat_array: np.ndarray,
                 mf_dim: int = 16, # Reduzido de 32 para 16 (Mais leve)
                 mlp_dim: int = 32, # Reduzido de 64 para 32 (Mais leve)
                 categoria_dim: int = 8,
                 hidden_dims = [128, 64], # Removida a camada de 256 para acelerar a CPU
                 dropout: float = 0.1):
        super().__init__()
        
        self.user_mf_embed = nn.Embedding(n_users, mf_dim)
        self.item_mf_embed = nn.Embedding(n_items, mf_dim)
        self.user_mlp_embed = nn.Embedding(n_users, mlp_dim)
        self.item_mlp_embed = nn.Embedding(n_items, mlp_dim)
        self.cat_mlp_embed = nn.Embedding(n_categorias, categoria_dim)
        
        self.register_buffer("item_categoria_idx", torch.tensor(item_to_cat_array, dtype=torch.long))
        
        in_dim = mlp_dim * 2 + categoria_dim
        mlp_layers = []
        for h_dim in hidden_dims:
            mlp_layers += [nn.Linear(in_dim, h_dim), nn.BatchNorm1d(h_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h_dim
        self.mlp = nn.Sequential(*mlp_layers)
        
        self.prediction_layer = nn.Linear(mf_dim + hidden_dims[-1], 1)

    def _forward_branch(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        user_mf = self.user_mf_embed(user_idx)
        item_mf = self.item_mf_embed(item_idx)
        gmf_vector = user_mf * item_mf
        
        user_mlp = self.user_mlp_embed(user_idx)
        item_mlp = self.item_mlp_embed(item_idx)
        cat_idx = self.item_categoria_idx[item_idx]
        cat_mlp = self.cat_mlp_embed(cat_idx)
        
        mlp_vector = torch.cat([user_mlp, item_mlp, cat_mlp], dim=-1)
        mlp_vector = self.mlp(mlp_vector)
        
        fusion = torch.cat([gmf_vector, mlp_vector], dim=-1)
        return self.prediction_layer(fusion).squeeze(-1)

    def forward(self, user_idx: torch.Tensor, pos_item_idx: torch.Tensor, neg_item_idx: torch.Tensor = None):
        if neg_item_idx is not None:
            return self._forward_branch(user_idx, pos_item_idx), self._forward_branch(user_idx, neg_item_idx)
        return self._forward_branch(user_idx, pos_item_idx)


class WeightedBPRLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pos_scores, neg_scores, weights):
        return - (weights * torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10)).mean()

In [51]:
# ==============================================================================
# 7. EXECUÇÃO DO TREINAMENTO OTIMIZADO PARA CPU
# ==============================================================================
# Definindo a constante global para manter sincronia com as assinaturas
N_RECS = 10 

def executar_treinamento_neumf_cpu(df_train, item_pop, item_cat_vector, num_users, num_items, num_cats):
    # Instancia o dataset com negativos pré-calculados
    train_dataset = RetailRocketBPRDatasetCPU(df_interactions=df_train, item_popularity=item_pop, n_items=num_items)
    
    # Batch size aumentado para 4096: Maximiza a eficiência do processamento linear na CPU
    train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True, num_workers=0)

    model = NeuMF_RetailRocket_CPU(n_users=num_users, n_items=num_items, n_categorias=num_cats, item_to_cat_array=item_cat_vector).to(device)
    criterion = WeightedBPRLoss()
    
    # Otimizador AdamW padrão
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    print("Iniciando treinamento na CPU...")
    model.train()
    for epoch in range(1, 4): # 3 épocas são suficientes para a validação do pipeline
        epoch_loss = 0
        for users, pos_items, neg_items, weights in train_loader:
            optimizer.zero_grad()
            pos_scores, neg_scores = model(users, pos_items, neg_items)
            loss = criterion(pos_scores, neg_scores, weights)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        print(f"Época {epoch} Finalizada | Loss Média: {epoch_loss/len(train_loader):.4f}")
        
    return model

# Executa o treino antes de abrir o run para focar o tracking no modelo pronto
model_treinado = executar_treinamento_neumf_cpu(
    df_train=df_train_pandas,
    item_pop=item_popularity,
    item_cat_vector=item_to_category_vector,
    num_users=num_users,
    num_items=num_items,
    num_cats=num_cats
)

Pré-gerando amostras negativas para otimizar a CPU...
Iniciando treinamento na CPU...
Época 1 Finalizada | Loss Média: 0.8195
Época 2 Finalizada | Loss Média: 0.8110
Época 3 Finalizada | Loss Média: 0.8079


In [52]:
# ==============================================================================
# 8. INFERÊNCIA COERENTE (RE-RANKING)
# ==============================================================================

def recomendar_mlp(visitor_id, model, candidatos_originais, idx_to_item, user_to_idx, item_to_idx, n=10):
    if len(candidatos_originais) == 0 or visitor_id not in user_to_idx:
        return []

    user_idx = user_to_idx[visitor_id]
    candidatos_validos = [item_to_idx[c] for c in candidatos_originais if c in item_to_idx]
    
    if len(candidatos_validos) == 0:
        return []

    model.eval()
    with torch.no_grad():
        user_tensor = torch.full((len(candidatos_validos),), user_idx, dtype=torch.long, device=device)
        item_tensor = torch.tensor(candidatos_validos, dtype=torch.long, device=device)
        scores = model(user_tensor, item_tensor).cpu().numpy()

    top_n = min(n, len(candidatos_validos))
    top_idxs_local = np.argsort(scores)[::-1][:top_n]
    top_item_idxs = [candidatos_validos[i] for i in top_idxs_local]

    return [idx_to_item[i] for i in top_item_idxs]

In [53]:
# ==============================================================================
# 8.a EXTRAÇÃO ULTRA-RÁPIDA DE CANDIDATOS E AVALIAÇÃO
# ==============================================================================

usuarios_teste_python = list(gt_dict.keys())

print("Construindo dicionário otimizado de candidatos por usuário...")
top_100_populares = [item for item, _ in sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)[:100]]
itens_treino_por_usuario = df_train_pandas.groupby('visitorid')['item_idx'].apply(list).to_dict()

resultados_experimentos = {}

print("Calculando métricas para o modelo NeuMF (MLP)...")

def recommend_wrapper_mlp_fast(user_id, k=10, **kwargs):
    mdl = kwargs.get('model')
    i_to_item = kwargs.get('idx_to_item')
    u_to_idx = kwargs.get('user_to_idx')
    i_to_idx = kwargs.get('item_to_idx')
    
    itens_usuario = itens_treino_por_usuario.get(user_id, [])
    candidatos_internos = list(set(itens_usuario + top_100_populares))
    candidatos_array = np.array(candidatos_internos, dtype=np.int64)
    candidatos_originais = np.array([i_to_item[idx] for idx in candidatos_array if idx in i_to_item])
    
    return recomendar_mlp(
        visitor_id=user_id,
        model=mdl,
        candidatos_originais=candidatos_originais,
        idx_to_item=i_to_item,
        user_to_idx=u_to_idx,
        item_to_idx=i_to_idx,
        n=k
    )

# Executa a avaliação do sistema
resultados_experimentos["NeuMF (MLP Otimizada)"] = avaliar_sistema_recomendacao(
    recommend_fn=recommend_wrapper_mlp_fast, 
    test_users=usuarios_teste_python, 
    gt_dict=gt_dict, 
    n_items_total=num_items, 
    k=N_RECS,
    model=model_treinado,
    idx_to_item=idx_to_item,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx
)

metricas_finais = resultados_experimentos["NeuMF (MLP Otimizada)"]

Construindo dicionário otimizado de candidatos por usuário...
Calculando métricas para o modelo NeuMF (MLP)...


In [54]:
# ==============================================================================
# 8.a EXTRAÇÃO ULTRA-RÁPIDA DE CANDIDATOS E AVALIAÇÃO
# ==============================================================================

usuarios_teste_python = list(gt_dict.keys())

print("Construindo dicionário otimizado de candidatos por usuário...")
top_100_populares = [item for item, _ in sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)[:100]]
itens_treino_por_usuario = df_train_pandas.groupby('visitorid')['item_idx'].apply(list).to_dict()

resultados_experimentos = {}

print("Calculando métricas para o modelo NeuMF (MLP)...")

def recommend_wrapper_mlp_fast(user_id, k=10, **kwargs):
    mdl = kwargs.get('model')
    i_to_item = kwargs.get('idx_to_item')
    u_to_idx = kwargs.get('user_to_idx')
    i_to_idx = kwargs.get('item_to_idx')
    
    itens_usuario = itens_treino_por_usuario.get(user_id, [])
    candidatos_internos = list(set(itens_usuario + top_100_populares))
    candidatos_array = np.array(candidatos_internos, dtype=np.int64)
    candidatos_originais = np.array([i_to_item[idx] for idx in candidatos_array if idx in i_to_item])
    
    return recomendar_mlp(
        visitor_id=user_id,
        model=mdl,
        candidatos_originais=candidatos_originais,
        idx_to_item=i_to_item,
        user_to_idx=u_to_idx,
        item_to_idx=i_to_idx,
        n=k
    )

# Executa a avaliação do sistema
resultados_experimentos["NeuMF (MLP Otimizada)"] = avaliar_sistema_recomendacao(
    recommend_fn=recommend_wrapper_mlp_fast, 
    test_users=usuarios_teste_python, 
    gt_dict=gt_dict, 
    n_items_total=num_items, 
    k=N_RECS,
    model=model_treinado,
    idx_to_item=idx_to_item,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx
)

metricas_finais = resultados_experimentos["NeuMF (MLP Otimizada)"]

Construindo dicionário otimizado de candidatos por usuário...
Calculando métricas para o modelo NeuMF (MLP)...


In [55]:
# ==============================================================================
# MONITORAMENTO MLFLOW - ENCAPSULAMENTO DE LOGS E ARTEFATOS
# ==============================================================================
import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature

# Definição do Lineage usando o dataframe mapeado final
dataset_mlp = mlflow.data.from_pandas(df_train_pandas, name="dataset_RetailRocket_Events_and_Properties")

print("Registrando corrida e artefatos no MLflow...")
with mlflow.start_run(run_name="Neural-NeuMF-MLP") as run:     
    
    mlflow.log_input(dataset_mlp, context="training/test")  
    
    # 1. Tags identificadoras (Mantendo a simetria exata com as dos baselines)
    mlflow.set_tags({
        "model_type": "NeuMF_Neural_Collaborative_Filtering",
        "model_architecture": "GMF_plus_MLP_with_Embeddings",
        "framework": "pytorch",
        "phase": "neural_model",
        "dataset_name": "RetailRocket_Events_and_Properties",
        "dataset_size": f"{len(df_train_pandas)}",
        "feature_set": "Implicit_Feedback_with_Category_Embeddings"
    })

    # 2. Hiperparâmetros do Modelo e do Treino
    mlflow.log_params({
        "model.mf_dim": 16,
        "model.mlp_dim": 32,
        "model.categoria_dim": 8,
        "model.hidden_layers": "[128, 64]",
        "model.dropout_rate": 0.1,
        "train.optimizer": "AdamW",
        "train.epochs": 3,
        "train.batch_size": 4096,
        "recommendation.top_k": N_RECS,
        "data.test_users_count": len(usuarios_teste_python)
    })
    
    # 3. Métricas extraídas corrigidas para bater com as chaves de retorno do dicionário
    mlflow.log_metrics({
        "eval.precision_at_10": metricas_finais[f"Precision@{N_RECS}"],
        "eval.recall_at_10": metricas_finais[f"Recall@{N_RECS}"],
        "eval.ndcg_at_10": metricas_finais[f"NDCG@{N_RECS}"],
        "eval.catalog_coverage": metricas_finais["Coverage"]
    })

    # 4. Assinatura Padronizada da API
    sample_user = np.array([12345], dtype=np.int64)
    sample_output = np.array(list(idx_to_item.values())[:N_RECS], dtype=np.int64)
    
    signature = infer_signature(
        model_input={"visitor_id": sample_user},
        model_output=sample_output
    )
        
    # 5. Salvamento estruturado de metadados de teste
    lista_usuarios_teste = [int(uid) for uid in gt_dict.keys()]
    pop_dict = {"test_users": lista_usuarios_teste}
    mlflow.log_dict(pop_dict, artifact_file="neural-mlp/mlp-users.json")
    
    # 6. Salvamento binário dos pesos do PyTorch no Run corrente
    mlflow.pytorch.log_model(
        pytorch_model=model_treinado,
        artifact_path="modelo-neumf-retailrocket",
        signature=signature,
        registered_model_name="neumf-mlp-recommendation"
    )
    
print("Corrida do MLflow para o modelo Neural finalizada e registrada com sucesso!")


# ==============================================================================
# 9. EXIBIÇÃO DA TABELA COMPARATIVA FINAL
# ==============================================================================

df_metricas = pd.DataFrame.from_dict(resultados_experimentos, orient="index")

df_metricas_display = df_metricas.copy()
for col in df_metricas_display.columns:
    df_metricas_display[col] = df_metricas_display[col].apply(lambda x: f"{x * 100:.2f}%")

print("\n" + "─" * 20 + " TABELA COMPARATIVA FINAL " + "─" * 20)
print(df_metricas_display)

Registrando corrida e artefatos no MLflow...


c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/21 16:49:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 16:49:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickl

Corrida do MLflow para o modelo Neural finalizada e registrada com sucesso!

──────────────────── TABELA COMPARATIVA FINAL ────────────────────
                      Precision@10 Recall@10 NDCG@10 Coverage
NeuMF (MLP Otimizada)        0.76%     2.80%   1.66%    1.44%
